In [1]:
import torch
import torch.nn as nn
import math

# Positional Encoding
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x


# Multi-Head Attention
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, heads):
        super().__init__()

        self.d_model = d_model
        self.heads = heads

        self.head_dim = d_model // heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, q, k, v):

        N = q.shape[0]

        Q = self.q_linear(q)
        K = self.k_linear(k)
        V = self.v_linear(v)

        scores = torch.matmul(
            Q, K.transpose(-2, -1)
        ) / math.sqrt(self.d_model)

        attention = torch.softmax(scores, dim=-1)

        out = torch.matmul(attention, V)

        out = self.fc_out(out)

        return out


# Feed Forward Network
class FeedForward(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, 2048),
            nn.ReLU(),
            nn.Linear(2048, d_model)
        )

    def forward(self, x):
        return self.net(x)


# Encoder Layer
class EncoderLayer(nn.Module):

    def __init__(self, d_model, heads):
        super().__init__()

        self.attention = MultiHeadAttention(
            d_model, heads
        )

        self.norm1 = nn.LayerNorm(d_model)

        self.ff = FeedForward(d_model)

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        attn = self.attention(x, x, x)

        x = self.norm1(x + attn)

        ff = self.ff(x)

        x = self.norm2(x + ff)

        return x


# Transformer Encoder
class Transformer(nn.Module):

    def __init__(self, d_model, heads, layers):
        super().__init__()

        self.pos = PositionalEncoding(d_model)

        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, heads)
             for _ in range(layers)]
        )

    def forward(self, x):

        x = self.pos(x)

        for layer in self.layers:
            x = layer(x)

        return x


# Test Model
x = torch.rand(32, 10, 512)

model = Transformer(
    d_model=512,
    heads=8,
    layers=2
)

output = model(x)

print(output.shape)

torch.Size([32, 10, 512])
